# RAG from Scratch to Production — Episode 11/20
## The reranker threshold bug: one number, two ways to fail

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/11_reranker_threshold.ipynb)

A real bug from the [production app](https://github.com/JakkulaMahendhar/rag-chatbot)'s bug log (#31):

> A question about the leave policy showed **3 sources**: the right one, plus a chunk from a **completely unrelated document**, at **0% relevance**.

The cause: the reranker always returned exactly 3 chunks. A relevance threshold (`0.20`) existed in the config, but **nothing ever read it**. With only 1 relevant chunk available, the other 2 slots were filled with whatever scored next, and that junk went into the LLM's context too.

The fix: apply the 0.20 threshold before taking the top 3.

Episode 10 found that the reranker gives some **correct** answers very low scores. So this episode asks: **did the fix create the opposite bug?**

**In this episode:**
1. Reproduce the original bug.
2. Apply the fix, and count what it throws away.
3. Try every threshold, and look for a perfect one.
4. Try a different idea: confidence **bands** instead of one cutoff.

**No API key needed.**

## 0. Setup

Episode 10's handbook, plus an **unrelated** second document (kitchen and meeting-room rules), like the two-document setup in the bug report.

In [ ]:
%pip install -q sentence-transformers rank-bm25

In [1]:
import re, math
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

HANDBOOK = [
    "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week.",
    "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year. Book leave with form HR-12.",
    "Parental Leave\nNew parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected start date.",
    "Expenses\nThe meal allowance during business travel is capped at 40 USD per day. Travel expenses are claimed with form EXP-22 within 30 days. Claims above 500 USD need the finance director.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over. Requests are approved by your direct manager.",
    "Equipment\nEvery employee receives a ThinkPad X1 Carbon laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours.",
    "IT Support\nIf the VPN shows error E-4031, your access token has expired: reset it in the Okta portal. For error E-4032, contact the IT helpdesk.",
    "Security\nMulti-factor authentication is mandatory on all company accounts. Passwords must be at least 14 characters long and must never be reused.",
]
KITCHEN_RULES = [
    "Kitchen Rules\nLabel your food in the shared fridge with your name and date. The fridge is emptied every Friday at 4 pm.",
    "Coffee Machine\nDescale the coffee machine every Monday. Report broken machines to the facilities team.",
    "Meeting Rooms\nBook meeting rooms in the calendar app. Rooms not used within 10 minutes of the booking start are released.",
]
CHUNKS = HANDBOOK + KITCHEN_RULES

def title(chunk: str) -> str:
    return chunk.split("\n")[0]

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

def minmax(x):
    return (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.zeros_like(x)

def sigmoid(x: float) -> float:
    return 1 / (1 + math.exp(-x))

embedder = SentenceTransformer("all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
vectors = embedder.encode(CHUNKS, normalize_embeddings=True)
bm25 = BM25Okapi([tokenize(c) for c in CHUNKS])


def reranked(question: str):
    """Hybrid search picks 10 candidates, the reranker scores them (0–1, like production), best first."""
    hybrid = 0.8 * minmax(vectors @ embedder.encode(question, normalize_embeddings=True)) \
           + 0.2 * minmax(np.array(bm25.get_scores(tokenize(question))))
    shortlist = list(np.argsort(-hybrid)[:10])
    scores = reranker.predict([[question, CHUNKS[i]] for i in shortlist])
    return sorted(((title(CHUNKS[i]), sigmoid(float(s))) for i, s in zip(shortlist, scores)),
                  key=lambda x: -x[1])

## 1. Three kinds of question

A threshold has two jobs, so we need to test both:
- **Answerable** (14): the answer is in the handbook. The threshold must **keep** the right chunk.
- **Off-topic** (4): nothing to do with the company. The threshold must **reject** everything.
- **In-domain but unanswerable** (6): they *sound* like handbook questions, but the handbook doesn't answer them. The hardest kind: a similar-looking chunk always exists.

In [2]:
ANSWERABLE = [
    ("How many weeks of leave do new parents get?",          "Parental Leave"),
    ("How many vacation days carry over to next year?",      "Annual Leave"),
    ("Does the learning budget carry over to next year?",    "Learning Budget"),
    ("Who has to approve expense claims over 500 USD?",      "Expenses"),
    ("Who approves my course and conference requests?",      "Learning Budget"),
    ("What happens if my VPN access token expired?",         "IT Support"),
    ("How fast do I report a stolen phone?",                 "Equipment"),
    ("Which day must I be in the office?",                   "Remote Work"),
    ("Can I reuse an old password?",                         "Security"),
    ("What's the deadline for logging remote days?",         "Remote Work"),
    ("How long do I have to submit a travel expense claim?", "Expenses"),
    ("When must the parental leave form be submitted?",      "Parental Leave"),
    ("Can I save unused training money for next year?",      "Learning Budget"),
    ("What do I get for food when travelling for work?",     "Expenses"),
]
OFF_TOPIC = [
    "What is the capital of France?",
    "Who won the football World Cup in 2022?",
    "What is the dress code for the office party?",
    "How do I bake sourdough bread?",
]
UNANSWERABLE = [
    "How much is the gym allowance?",
    "How many sick days do I get per year?",
    "Can I bring my dog to the office?",
    "Is there a company pension plan?",
    "What is the bonus for referring a new hire?",
    "How much is the parking allowance?",
]

RESULTS = {q: reranked(q) for q in [q for q, _ in ANSWERABLE] + OFF_TOPIC + UNANSWERABLE}
print(f"Scored {len(RESULTS)} questions")

Scored 24 questions


## 2. The bug: always the top 3

In [3]:
def sources_before_fix(question):
    return RESULTS[question][:3]

for q in ["How many weeks of leave do new parents get?", "What is the capital of France?"]:
    print(f"Q: {q}")
    for name, score in sources_before_fix(q):
        print(f"   {score:6.1%}  {name}")
    print()

Q: How many weeks of leave do new parents get?
   100.0%  Parental Leave
     3.3%  Annual Leave
     0.0%  Remote Work
Q: What is the capital of France?
     0.0%  Remote Work
     0.0%  Annual Leave
     0.0%  Parental Leave


The bug exactly as reported: the right section at 100%, plus two unrelated sources at **3.3%** and **0.0%**. And an off-topic question still gets 3 "sources" handed to the LLM.

## 3. The fix: keep only chunks scoring at least 0.20

In [4]:
def evaluate(policy, label):
    """policy(question) -> list of (section, score) sent to the LLM."""
    kept = junk = 0
    dropped = []
    for q, answer in ANSWERABLE:
        sent = [name for name, _ in policy(q)]
        if answer in sent:
            kept += 1
        else:
            dropped.append(q)
        junk += sum(name != answer for name in sent)
    off = sum(len(policy(q)) > 0 for q in OFF_TOPIC)
    unanswerable = sum(len(policy(q)) > 0 for q in UNANSWERABLE)
    print(f"{label:<22} right chunk kept {kept:>2}/14   junk chunks sent {junk:>2}   "
          f"off-topic with chunks {off}/4   unanswerable with chunks {unanswerable}/6")
    return dropped


def threshold(t):
    return lambda q: [x for x in RESULTS[q] if x[1] >= t][:3]


evaluate(sources_before_fix, "Before the fix (top 3)")
dropped = evaluate(threshold(0.20), "After the fix (0.20)")

Before the fix (top 3) right chunk kept 14/14   junk chunks sent 28   off-topic with chunks 4/4   unanswerable with chunks 6/6
After the fix (0.20)   right chunk kept 11/14   junk chunks sent  0   off-topic with chunks 0/4   unanswerable with chunks 0/6


The fix works for what it was meant to fix: **zero** junk chunks, and off-topic questions get nothing.

But it now throws away the right answer for 3 answerable questions. Which ones, and why?

In [5]:
for q in dropped:
    answer = dict(ANSWERABLE)[q]
    ranking = RESULTS[q]
    position = [name for name, _ in ranking].index(answer)
    print(f"Q: {q}")
    print(f"   right section: {answer}, score {ranking[position][1]:.4f}, ranked #{position + 1}")

Q: Can I reuse an old password?
   right section: Security, score 0.0192, ranked #1
Q: Can I save unused training money for next year?
   right section: Learning Budget, score 0.0000, ranked #2
Q: What do I get for food when travelling for work?
   right section: Expenses, score 0.0324, ranked #1


- *"Can I reuse an old password?"* and *"food when travelling"*: the reranker **ranked the right section first**, but scored it below 0.20. The fix throws away a correct answer, and the bot says *"I don't have enough information"*.
- *"unused training money"*: a different problem. The reranker ranked the **wrong** section first (Episode 10). No threshold fixes a wrong ranking.

So the fix traded one failure (junk in the answer) for another (refusing answerable questions). Is there a better number?

## 4. Search for the perfect threshold

In [6]:
for t in [0.5, 0.20, 0.05, 0.01, 0.001]:
    evaluate(threshold(t), f"threshold {t}")

threshold 0.5          right chunk kept 11/14   junk chunks sent  0   off-topic with chunks 0/4   unanswerable with chunks 0/6
threshold 0.2          right chunk kept 11/14   junk chunks sent  0   off-topic with chunks 0/4   unanswerable with chunks 0/6
threshold 0.05         right chunk kept 11/14   junk chunks sent  0   off-topic with chunks 0/4   unanswerable with chunks 1/6
threshold 0.01         right chunk kept 13/14   junk chunks sent  1   off-topic with chunks 0/4   unanswerable with chunks 1/6
threshold 0.001        right chunk kept 13/14   junk chunks sent  5   off-topic with chunks 0/4   unanswerable with chunks 3/6


Lower thresholds keep more correct answers, but let in junk and unanswerable questions. Higher ones do the opposite. **No setting gets everything right.** Here's why:

In [7]:
rows = []
for q, answer in ANSWERABLE:
    best_name, best_score = RESULTS[q][0]
    rows.append((best_score, "✅ right section" if best_name == answer else "❌ wrong section", q))
for q in UNANSWERABLE:
    rows.append((RESULTS[q][0][1], "⚠️ not in handbook", q))
for q in OFF_TOPIC:
    rows.append((RESULTS[q][0][1], "🚫 off-topic", q))

for score, kind, q in sorted(rows, reverse=True):
    print(f"{score:.4f}  {kind:<20} {q}")

0.9999  ✅ right section      How many weeks of leave do new parents get?
0.9995  ✅ right section      What happens if my VPN access token expired?
0.9995  ✅ right section      When must the parental leave form be submitted?
0.9950  ✅ right section      Who approves my course and conference requests?
0.9874  ✅ right section      How many vacation days carry over to next year?
0.9873  ✅ right section      Which day must I be in the office?
0.9870  ✅ right section      How long do I have to submit a travel expense claim?
0.9696  ✅ right section      Does the learning budget carry over to next year?
0.9684  ✅ right section      Who has to approve expense claims over 500 USD?
0.9321  ✅ right section      What's the deadline for logging remote days?
0.7197  ✅ right section      How fast do I report a stolen phone?
0.1326  ⚠️ not in handbook   How many sick days do I get per year?
0.0324  ✅ right section      What do I get for food when travelling for work?
0.0192  ✅ right section      Can I 

Look at the middle of the list:
- *"How many **sick days** do I get?"* isn't in the handbook. Its best chunk (Annual Leave) scores **0.1326**.
- *"Can I reuse an old password?"* is answered by the Security section, which scores **0.0192**.

**A wrong chunk scores 7× higher than a right one.** Any single cutoff that keeps the password answer also lets the sick-days question through, and the LLM might answer it with the annual leave rules. One number can't separate them.

The reranker's scores are excellent at the extremes: off-topic questions score 0.0000, and most correct answers score above 0.7. It's the **middle** that's uncertain.

## 5. A different fix: three bands instead of one cutoff

Instead of a yes/no cutoff, treat the middle as what it is, **uncertain**:

| Best score | Band | What the bot does |
|---|---|---|
| ≥ 0.20 | **Confident** | Answer normally |
| 0.001 – 0.20 | **Low confidence** | Answer, but say so: "I'm not sure. The closest section is *X*" |
| < 0.001 | **Not found** | "That isn't in the documents" |

In [8]:
def band(score: float) -> str:
    if score >= 0.20:
        return "confident"
    return "low confidence" if score >= 0.001 else "not found"


groups = {
    "answerable":   [(q, a) for q, a in ANSWERABLE],
    "off-topic":    [(q, None) for q in OFF_TOPIC],
    "unanswerable": [(q, None) for q in UNANSWERABLE],
}
for group, questions in groups.items():
    counts = {}
    for q, answer in questions:
        best_name, best_score = RESULTS[q][0]
        key = band(best_score)
        if answer:
            key += " ✅" if best_name == answer else " ❌ wrong section"
        counts[key] = counts.get(key, 0) + 1
    print(f"{group:<13} {counts}")

answerable    {'confident ✅': 11, 'low confidence ✅': 2, 'low confidence ❌ wrong section': 1}
off-topic     {'not found': 4}
unanswerable  {'low confidence': 3, 'not found': 3}


- **Confident**: 11 answers, **all correct**. No confident wrong answers.
- **Not found**: all 4 off-topic questions, and half the unanswerable ones.
- **Low confidence**: the uncertain middle (2 correct answers, 1 wrong ranking, 3 unanswerable). Instead of silently refusing or confidently guessing, the bot is honest that it isn't sure, and shows the user where to look.

The exact band edges (0.20, 0.001) were chosen on these 24 questions. For your documents, pick them the same way: collect real questions, score them, and look where the answerable and unanswerable ones overlap.

## 6. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) (`app/services/reranker.py`, after the fix):
- Drops chunks scoring below **0.20**, then keeps the top 3. If nothing passes, the LLM gets no context and replies *"I don't have enough information"*.
- So correct answers that the reranker scores low (here: 2 of 14) are refused. The frontend already shows an answer-quality badge from the **search** score. Driving it from the **reranker** score, with a low-confidence band, would make those cases visible instead of refused.

## Recap

| Approach | Junk sent to the LLM | Correct answers refused | Unanswerable passed through |
|---|---|---|---|
| Always top 3 (the bug) | Lots | 0 | All |
| One threshold (0.20) | None | 2 of 14 (+1 wrong ranking) | 0 of 6 |
| **Three bands** | None marked confident | **None refused, 3 flagged** | **None answered confidently** |

**The rule:** a threshold has two failure modes, so test both: questions it must answer, and questions it must refuse. If they overlap, don't pretend one number can split them. Show the uncertainty.

**Next, Episode 12: context engineering.** The chunks that pass go into the prompt. How many, in what order, and how much text?

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)